In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_community.document_loaders import CSVLoader
from dotenv import load_dotenv

load_dotenv()

file = 'petshop.csv'
loader = CSVLoader(file_path=file)
docs = loader.load()

# 1. Setup Data & Vector Store (Explicit & Lightweight)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 2. The Modern "Stuff" RAG Chain
# We explicitly handle the document separation logic here
def format_docs(docs):
    return "<<<<>>>>>".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context:
{context}

Question: {question}
""")

# The pipe operator (|) replaces RetrievalQA
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 3. Modern Example Generation (Replaces QAGenerateChain)
# We use a prompt that asks for JSON to make parsing reliable
gen_prompt = ChatPromptTemplate.from_template("""
Your task is to write a query and answer pair from the following document.
Return the result in JSON format with keys 'query' and 'answer'.

Document: {doc}
""")

# This chain generates one example per document
example_gen_chain = gen_prompt | llm | JsonOutputParser()

# Generate examples for the first 5 docs
new_examples = example_gen_chain.batch([{"doc": d.page_content} for d in docs[:5]])

print(new_examples[0]) # {'query': '...', 'answer': '...'}

{'query': 'What is the sale price of the cat?', 'answer': 'The sale price of the cat is $450.09.'}


In [4]:
new_examples

[{'query': 'What is the sale price of the cat?',
  'answer': 'The sale price of the cat is $450.09.'},
 {'query': 'What is the sale price of the dog?',
  'answer': 'The sale price of the dog is 666.66.'},
 {'query': 'What is the sale price of the parrot and how many were sold?',
  'answer': 'The sale price of the parrot is $50.00 and 2 were sold.'},
 {'query': 'What is the sale price of the hamster and on what date was it sold?',
  'answer': 'The sale price of the hamster is $60.60 and it was sold on 2018-06-11.'},
 {'query': 'What is the sale price of the goldfish?',
  'answer': 'The sale price of the goldfish is 48.48.'}]

In [8]:
retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x1762a4440>, search_kwargs={})

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Initialize the "Judge" LLM
eval_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 2. Define a clear Evaluation Prompt
# This replaces the hidden prompt inside QAEvalChain
eval_prompt = ChatPromptTemplate.from_template("""
You are an expert grader. Given a Question, the Ground Truth answer, and a Predicted Answer, 
determine if the Predicted Answer is correct.

Question: {query}
Ground Truth: {answer}
Prediction: {result}

Return your response in JSON format with two keys:
"grade": "CORRECT" or "INCORRECT"
"reasoning": "A brief explanation of why the grade was given."
""")

# 3. Create the Evaluator Chain
evaluator = eval_prompt | eval_llm | JsonOutputParser()

# 4. Run Evaluation (Replaces eval_chain.evaluate)
# Use .batch for efficiency
graded_outputs = evaluator.batch(example_gen_chain)

# 5. Display Results
for i, (eg, grade_info) in enumerate(zip(new_examples, graded_outputs)):
    print(f"Example {i}:")
    print(f"Question: {predictions[i]['query']}")
    print(f"Real Answer: {predictions[i]['answer']}")
    print(f"Predicted Answer: {predictions[i]['result']}")
    print(f"Grade: {grade_info['grade']}")
    print(f"Reasoning: {grade_info['reasoning']}\n")

TypeError: object of type 'RunnableSequence' has no len()